# 04 - Incremental Sales Load
Watermark-based incremental processing with Delta MERGE.

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

base_path = '/mnt/retail'
source_path = f'{base_path}/input/sales.csv'
target_path = f'{base_path}/silver/sales'
last_processed_date = '2026-01-12'  # Production: read from a control table


In [ ]:
incoming = (spark.read.option('header', True).option('inferSchema', True).csv(source_path)
    .withColumn('sale_date', F.to_date('sale_date')))

incremental = (incoming
    .filter(F.col('sale_date') > F.to_date(F.lit(last_processed_date)))
    .dropDuplicates(['transaction_id'])
    .filter(F.col('quantity') > 0)
    .withColumn('total_amount', F.col('quantity') * F.col('unit_price')))

In [ ]:
target = DeltaTable.forPath(spark, target_path)
(target.alias('t')
 .merge(incremental.alias('s'), 't.transaction_id = s.transaction_id')
 .whenMatchedUpdateAll()
 .whenNotMatchedInsertAll()
 .execute())

### Interview points
- Watermark processing avoids scanning all historical records every run.
- Delta MERGE supports an idempotent upsert pattern.
- ADF can pass the watermark to the Databricks notebook.
- Update the watermark only after successful processing.